In [35]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_leads = 300

# Raw uncleaned variations
first_names = [" john", "SARAH", "mIchael", "Elena ", "Ahmadi", " David"]
last_names = ["SMITH", "jenkins ", "Chang", "rostova", "HASSAN", "miller"]
domains = ["gmail.com", "techcorp.io", "enterprise.com", "yahoo.com", "innovate.co"]
titles = [
    "CEO",
    "Chief Executive Officer",
    "VP of Sales",
    "Sales Rep",
    "Head of Revenue",
    "Data Analyst",
    "Founder",
]
industries = ["Software", "Healthcare", "Finance", "Retail", "Manufacturing"]
company_sizes = ["1-10", "11-50", "51-200", "201-500", "500+"]

data = []
for i in range(n_leads):
    fn = np.random.choice(first_names)
    ln = np.random.choice(last_names)
    domain = np.random.choice(domains)
    email = f"{fn.strip().lower()}.{ln.strip().lower()}@{domain}"

    data.append(
        {
            "lead_id": f"LEAD-{100 + i}",
            "first_name": fn,
            "last_name": ln,
            "email": email,
            "job_title": np.random.choice(titles),
            "industry": np.random.choice(industries),
            "company_size": np.random.choice(company_sizes),
        }
    )

# Create DataFrame & introduce duplicate emails / missing values
df_messy = pd.DataFrame(data)
df_messy.loc[::10, "email"] = df_messy.loc[0, "email"]  # Inject duplicates
df_messy.loc[::15, "company_size"] = None  # Inject missing values

df_messy.to_csv("messy_leads_input.csv", index=False)
print(
    "✅ Created 'messy_leads_input.csv' with uncleaned names, titles, and missing data."
)

✅ Created 'messy_leads_input.csv' with uncleaned names, titles, and missing data.


In [36]:
df = pd.read_csv("/content/messy_leads_input.csv")
df.info()
print(df.isna().sum())
df.duplicated().sum().sum()
#

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   lead_id       300 non-null    object
 1   first_name    300 non-null    object
 2   last_name     300 non-null    object
 3   email         300 non-null    object
 4   job_title     300 non-null    object
 5   industry      300 non-null    object
 6   company_size  280 non-null    object
dtypes: object(7)
memory usage: 16.5+ KB
lead_id          0
first_name       0
last_name        0
email            0
job_title        0
industry         0
company_size    20
dtype: int64


np.int64(0)

In [37]:
import pandas as pd

# 1. Load the messy dataset
df = pd.read_csv("messy_leads_input.csv")

print(f"Original record count: {len(df)}")

# 2. Clean Text Formatting (Strip whitespace & capitalize names properly)
df["first_name"] = df["first_name"].astype(str).str.strip().str.title()
df["last_name"] = df["last_name"].astype(str).str.strip().str.title()
df["full_name"] = df["first_name"] + " " + df["last_name"]

# 3. Clean Emails & Extract Domain
df["email"] = df["email"].astype(str).str.strip().str.lower()
df["domain"] = df["email"].apply(
    lambda x: x.split("@")[-1] if "@" in x else "unknown"
)

# 4. Flag Free Email Providers vs. Business Domains
free_providers = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com"]
df["is_business_email"] = ~df["domain"].isin(free_providers)

# 5. Handle Missing Values
df["company_size"] = df["company_size"].fillna("Unknown")

# 6. Deduplicate Rows based on Email
df_clean = df.drop_duplicates(subset=["email"]).copy()

print(f"Cleaned record count (after deduplication): {len(df_clean)}")
print("\n--- SAMPLE CLEANED LEADS ---")
print(
    df_clean[
        [
            "full_name",
            "email",
            "domain",
            "is_business_email",
            "job_title",
            "company_size",
        ]
    ].head()
)

Original record count: 300
Cleaned record count (after deduplication): 136

--- SAMPLE CLEANED LEADS ---
       full_name                         email          domain  \
0   Elena Hassan   elena.hassan@enterprise.com  enterprise.com   
1  Michael Chang  michael.chang@enterprise.com  enterprise.com   
2   David Hassan      david.hassan@techcorp.io     techcorp.io   
3   Ahmadi Smith        ahmadi.smith@yahoo.com       yahoo.com   
4     John Smith     john.smith@enterprise.com  enterprise.com   

   is_business_email                job_title company_size  
0               True          Head of Revenue      Unknown  
1               True          Head of Revenue       51-200  
2               True                Sales Rep      201-500  
3              False  Chief Executive Officer      201-500  
4               True              VP of Sales      201-500  


In [38]:
df.info()
df.isna().sum()
df.duplicated().sum().sum()
print(len(df))
print(len(df_clean))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   lead_id            300 non-null    object
 1   first_name         300 non-null    object
 2   last_name          300 non-null    object
 3   email              300 non-null    object
 4   job_title          300 non-null    object
 5   industry           300 non-null    object
 6   company_size       300 non-null    object
 7   full_name          300 non-null    object
 8   domain             300 non-null    object
 9   is_business_email  300 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 21.5+ KB
300
136


In [39]:
def calculate_lead_score(row):
    score = 0

    # 1. Job Title Authority Scoring
    title = str(row["job_title"]).lower()
    if any(term in title for term in ["ceo", "chief", "founder"]):
        score += 30
    elif any(term in title for term in ["vp", "head", "director"]):
        score += 20
    else:
        score += 10

    # 2. Company Size Scoring
    size = str(row["company_size"])
    if size in ["201-500", "500+"]:
        score += 30
    elif size == "51-200":
        score += 20
    elif size == "11-50":
        score += 10
    else:
        score += 5

    # 3. Industry Fit Scoring
    industry = str(row["industry"])
    if industry in ["Software", "Finance"]:
        score += 20
    else:
        score += 10

    # 4. Email Quality Scoring
    if row["is_business_email"]:
        score += 20
    else:
        score -= 10

    return score


# Apply scoring function across each row
df_clean["lead_score"] = df_clean.apply(calculate_lead_score, axis=1)


# Categorize leads into actionable sales tiers
def assign_tier(score):
    if score >= 80:
        return "Hot (Direct Sales Call)"
    elif score >= 50:
        return "Warm (Email Campaign)"
    else:
        return "Cold (Archive / Ignore)"


df_clean["lead_tier"] = df_clean["lead_score"].apply(assign_tier)

# Sort dataset by highest score first
df_clean = df_clean.sort_values(by="lead_score", ascending=False)

# Export cleaned & scored output to CSV
df_clean.to_csv("scored_leads_output.csv", index=False)

print("--- LEAD TIER DISTRIBUTION ---")
print(df_clean["lead_tier"].value_counts())

print("\n--- TOP 5 HIGHEST SCORING LEADS ---")
print(
    df_clean[
        [
            "full_name",
            "email",
            "job_title",
            "company_size",
            "lead_score",
            "lead_tier",
        ]
    ].head()
)
print(len(df_clean))
print(len(df_clean.drop_duplicates(subset=["email"])))
print(len(df))

--- LEAD TIER DISTRIBUTION ---
lead_tier
Warm (Email Campaign)      61
Hot (Direct Sales Call)    39
Cold (Archive / Ignore)    36
Name: count, dtype: int64

--- TOP 5 HIGHEST SCORING LEADS ---
           full_name                         email    job_title company_size  \
129     David Miller   david.miller@enterprise.com          CEO         500+   
187      David Smith    david.smith@enterprise.com      Founder         500+   
36     David Rostova  david.rostova@enterprise.com  VP of Sales         500+   
42   Michael Rostova   michael.rostova@techcorp.io          CEO      201-500   
16       Elena Chang    elena.chang@enterprise.com          CEO       51-200   

     lead_score                lead_tier  
129         100  Hot (Direct Sales Call)  
187         100  Hot (Direct Sales Call)  
36           90  Hot (Direct Sales Call)  
42           90  Hot (Direct Sales Call)  
16           90  Hot (Direct Sales Call)  
136
136
300


In [42]:
import pandas as pd

# Load our cleaned and scored dataframe
# (Assuming df_clean already has 'lead_score' calculated from earlier)

# New: Calculate Lead Score based on various factors
def calculate_lead_score(row):
    score = 0
    # Factor 1: Business Email (higher score for business emails)
    if row["is_business_email"]:
        score += 30

    # Factor 2: Company Size (larger companies get higher scores)
    company_size_map = {
        "500+": 40,
        "201-500": 30,
        "51-200": 20,
        "11-50": 10,
        "1-10": 5,
        "Unknown": 0,
    }
    score += company_size_map.get(row["company_size"], 0)

    # Factor 3: Job Title (executive roles get higher scores)
    job_title = str(row["job_title"]).lower()
    if any(keyword in job_title for keyword in ["ceo", "chief executive officer", "founder", "head of revenue"]):
        score += 30
    elif "vp of sales" in job_title:
        score += 20
    elif "sales rep" in job_title:
        score += 10
    elif "data analyst" in job_title:
        score += 5

    return score

df_clean["lead_score"] = df_clean.apply(calculate_lead_score, axis=1)

# 1. Update Tier Names to Reflect Priority Calling Order
def assign_calling_priority(score):
    if score >= 80:
        return "Priority A - High Value (Call First)"
    elif score >= 50:
        return "Priority B - Medium Value (Call Mid-Day)"
    else:
        return "Priority C - Low Value (Call Last / Batch Email)"


df_clean["calling_priority"] = df_clean["lead_score"].apply(
    assign_calling_priority
)

# 2. Sort all 136 leads by rank (1 to 136)
df_clean = df_clean.sort_values(
    by=["lead_score", "full_name"], ascending=[False, True]
)
df_clean["call_rank"] = range(1, len(df_clean) + 1)

# 3. Export the complete Master Call Sheet containing ALL 136 leads
output_cols = [
    "call_rank",
    "full_name",
    "email",
    "job_title",
    "company_size",
    "lead_score",
    "calling_priority",
]
df_clean[output_cols].to_csv("master_daily_call_sheet.csv", index=False)

print("✅ Master Call Sheet generated: 'master_daily_call_sheet.csv'")
print(f"Total leads assigned to rep: {len(df_clean)}")

print("\n--- SAMPLE MASTER CALL SHEET (TOP 10 LEADS) ---")
print(df_clean[output_cols].head(10).to_string(index=False))

print("\n--- SAMPLE MASTER CALL SHEET (BOTTOM 5 LEADS) ---")
print(df_clean[output_cols].tail(5).to_string(index=False))

✅ Master Call Sheet generated: 'master_daily_call_sheet.csv'
Total leads assigned to rep: 136

--- SAMPLE MASTER CALL SHEET (TOP 10 LEADS) ---
 call_rank      full_name                        email       job_title company_size  lead_score                     calling_priority
         1  David Jenkins david.jenkins@enterprise.com             CEO         500+         100 Priority A - High Value (Call First)
         2   David Miller  david.miller@enterprise.com             CEO         500+         100 Priority A - High Value (Call First)
         3    David Smith   david.smith@enterprise.com         Founder         500+         100 Priority A - High Value (Call First)
         4   Elena Hassan     elena.hassan@innovate.co Head of Revenue         500+         100 Priority A - High Value (Call First)
         5   John Rostova  john.rostova@enterprise.com Head of Revenue         500+         100 Priority A - High Value (Call First)
         6 Michael Miller   michael.miller@innovate.co     